# 15. Proyección del cargo

Proyección del cargo por potencia de la próxima estación a partir del pronóstico y sus bandas (tarea 17 del backlog, HU-06). Es el cálculo que muestra la pantalla de consulta de la declaración.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd 
import numpy as np
from pathlib import Path
from modelado import fechas_estacion, calcular_grados, filtrar_entrenamiento, calcular_temperatura_media, armar_futuro, pronosticar_cuantiles, asignar_estaciones, calcular_maximos_estacionales
from regulacion import proyectar_cargo

In [3]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])

## 1. Pronóstico mensual del verano 2026

Se pronostican los seis meses del verano 2026 con los datos anteriores a su inicio, igual que en el pronóstico retrospectivo.

In [4]:
inicio, fin = fechas_estacion("VERANO", 2026)

In [5]:
pasado = serie[serie["MES"] < inicio].copy()
pasado = calcular_grados(pasado)

In [6]:
entrenamiento = filtrar_entrenamiento(pasado)
temp_media = calcular_temperatura_media(pasado)
futuro = armar_futuro(inicio, fin, temp_media)

In [7]:
tabla_ordenada = pronosticar_cuantiles(entrenamiento, futuro)

In [8]:
futuro[["Q10_MW", "Q50_MW", "Q90_MW"]] = tabla_ordenada[[0.1, 0.5, 0.9]].values
futuro[["MES", "Q10_MW", "Q50_MW", "Q90_MW"]]

,MES,Q10_MW,Q50_MW,Q90_MW
0,2026-11-01,38.038141,40.831556,44.566223
1,2026-12-01,38.764093,42.091490,45.491887
2,2027-01-01,39.733881,43.525924,46.545755
3,2027-02-01,39.128857,42.724561,45.956999
4,2027-03-01,38.185075,41.086567,44.753578
5,2027-04-01,36.470809,38.106152,42.563889


## 2. Cargo proyectado al declarar el piso

Para cada mes se calcula el cargo con los cuantiles 0,1, 0,5 y 0,9, y el cargo de la estación es la suma de los seis meses. Sumar el mismo cuantil de todos los meses supone que, si un mes queda alto, todos quedan altos. El rango resultante es más ancho y más conservador que el real.

In [9]:
serie = asignar_estaciones(serie)
maximos = calcular_maximos_estacionales(serie)

In [10]:
futuro = proyectar_cargo(futuro, maximos, "VERANO", 2026)
futuro[["MES", "Q10_MW", "Q50_MW", "Q90_MW", "PISO_MW", "HRP", "CARGO_Q10_USD", "CARGO_Q50_USD", "CARGO_Q90_USD"]]

,MES,Q10_MW,Q50_MW,Q90_MW,PISO_MW,HRP,CARGO_Q10_USD,CARGO_Q50_USD,CARGO_Q90_USD
0,2026-11-01,38.038141,40.831556,44.566223,39.82,380,217895.04,223430.27,243866.37
1,2026-12-01,38.764093,42.091490,45.491887,39.82,408,285939.46,302250.57,326668.14
2,2027-01-01,39.733881,43.525924,46.545755,39.82,386,270521.15,295697.72,316213.24
3,2027-02-01,39.128857,42.724561,45.956999,39.82,360,252299.52,270702.82,291183.55
4,2027-03-01,38.185075,41.086567,44.753578,39.82,408,233950.46,241391.80,262936.22
5,2027-04-01,36.470809,38.106152,42.563889,39.82,392,224775.94,224775.94,240264.64


In [11]:
cargo_estacion_cuantil = futuro[["CARGO_Q10_USD", "CARGO_Q50_USD", "CARGO_Q90_USD"]].sum()
cargo_estacion_cuantil

CARGO_Q10_USD    1485381.57
CARGO_Q50_USD    1558249.12
CARGO_Q90_USD    1681132.16
dtype: float64

## Conclusión

- El piso del verano 2026 es de 39,82 MW, el 90 % del máximo del verano 2025.
- Declarando el piso, el cargo proyectado de la estación es de 1,56 millones de dólares, con un rango de 1,49 millones a 1,68 millones.
- La banda inferior queda por debajo del piso en los seis meses, y en abril también el valor central. En esos casos el cargo se calcula sobre el piso, así que el extremo inferior del rango, 1,49 millones, es el cargo de pagar el piso toda la estación.